# 17 · Serving

A model saved by WarpRec can be put behind an HTTP API with one command and one
configuration file, without the dataset or the training configuration. This
guide trains three models on MovieLens-100K, serves them together and queries
them the way an application or an LLM agent would. It covers:

- what `meta.save_model` writes for serving, from the train pipeline and from
  the writer's `write_model`: seen items, histories, context vocabularies and
  the facts of the training run;
- the serving configuration (`serve.yml`): endpoints, item names and genres
  from `items.tsv`, unknown users, batching, the API key, and how the schema
  rejects a key it does not know;
- starting `warprec.serve` and the REST API: model cards, recommendations for
  a user, for an anonymous session and in a context, scoring your own
  candidates, item search, popular items, filters, and the errors;
- the MCP endpoint, called with an MCP client as an agent would;
- `--export`, which turns the same file into a Ray Serve config for a cluster.

Guide 10 covers the train pipeline, guide 5 contexts and guide 14 the
checkpoint file itself.

**Requirements:** `pip install "warprec[mcp]" jupyter` (the `mcp` extra
includes the `serving` one). Runs in five to eight minutes on a laptop CPU,
most of it training; the server listens on port 8117 while the guide runs.

In [1]:
import json
import logging
import re
import shutil
import signal
import subprocess
import sys
import time
import warnings
from pathlib import Path

import lightning as L
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()  # downloads and converts it as guide 1 shows
RUNS = Path("../runs/17-serving")
shutil.rmtree(RUNS, ignore_errors=True)
RUNS.mkdir(parents=True)

logger.disable("warprec")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")
ANSI = re.compile(r"\x1b\[[0-9;]*m")

## 1. Checkpoints to serve

Serving reads the `.pth` file that `meta.save_model: true` writes. Two models
come from one train-pipeline run: EASE, a general model fitted in closed form,
and SASRec, a sequential one trained for a few epochs. Each user's last rating
is the test set and the one before it the validation set.

In [2]:
print(Path("configs/train.yml").read_text())

# Two models to serve, trained and saved by the train pipeline: EASE (general,
# closed form) and SASRec (sequential). Each user's last interaction is the
# test set, the one before it the validation set. meta.save_model writes each
# best model to <local_experiment_path>/<dataset_name>/serialized/.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_leave_k_out
    k: 1
  validation_splitting:
    strategy: temporal_leave_k_out
    k: 1
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/17-serving/
models:
  EASE:
    meta: {save_model: true}
    l2: 200.0
  SASRec:
    meta: {save_model: true}
    optimization: {num_workers: 0}
    embedding_size: 32
    n_layers: 1
    n_heads: 1
    inner_size: 64
    dropout_prob: 0.2
    attn_dropout_prob: 0.2
    reg_weight: 0.0
    weight_decay: 0.0
    batch_

The pipeline runs on a private Ray instance (`general.ray_address: local`).
The notebook starts it as the command line would,
`python -m warprec.run -c configs/train.yml -p train`, with the log in a file.

In [3]:
with open(RUNS / "train.log", "w") as log:
    subprocess.run(
        [sys.executable, "-m", "warprec.run", "-c", "configs/train.yml", "-p", "train"],
        stdout=log,
        stderr=subprocess.STDOUT,
        check=True,
    )
serialized = RUNS / "ml-100k/serialized"
{p.name: f"{p.stat().st_size / 1e6:.1f} MB" for p in sorted(serialized.iterdir())}

{'EASE_WiseFoxOfForce.pth': '34.2 MB',
 'SASRec_FlamingoOfInevitableEngineering.pth': '0.9 MB'}

Each file is named after the model and a readable name derived from a hash of
its hyperparameters, so the same configuration always writes the same file
name; the serving configuration below refers to these names.

### 1.1 What a checkpoint carries for serving

Next to the weights and the id mappings (guide 14 takes the file apart), the
pipeline adds a `serving` payload built from the training data: everything a
server needs to answer without the dataset.

In [4]:
ease_file = torch.load(serialized / "EASE_WiseFoxOfForce.pth", weights_only=False)
sasrec_file = torch.load(
    serialized / "SASRec_FlamingoOfInevitableEngineering.pth", weights_only=False
)
print("EASE histories:", ease_file["serving"]["histories"])
payload = sasrec_file["serving"]
print(list(payload))
print(
    "seen:", type(payload["seen"]).__name__, payload["seen"].shape, payload["seen"].nnz
)
offsets = payload["histories"]["offsets"]
print("history length of users 0-4:", np.diff(offsets)[:5].tolist())

EASE histories: None
['seen', 'histories', 'context_maps', 'context_stats', 'training']
seen: csr_matrix (943, 1678) 98114
history length of users 0-4: [20, 20, 20, 20, 20]


- `seen` is the binary user × item training matrix. It masks what a user has
  already seen and ranks items by popularity for users the model does not know.
- `histories` holds each user's last `max_seq_len` (20) training items, packed
  into one array with per-user offsets. It lets a sequential model answer from
  a user id alone; EASE's is `None`.
- `context_maps` and `context_stats` are filled for context-aware models
  only (section 1.2).
- `training` records the run, and the model card serves it as is:

In [5]:
training = {k: v for k, v in payload["training"].items() if k != "trained_at"}
training

{'dataset': 'ml-100k',
 'n_users': 943,
 'n_items': 1678,
 'n_interactions': 98114,
 'evaluation': {'strategy': 'full'},
 'metrics': {'nDCG@10': 0.07221100479364395,
  'HitRate@10': 0.14027629792690277}}

The metrics are the test results the pipeline logged, averaged the way the
results table reports them; `trained_at` (left out above) is the save time.

### 1.2 A context-aware model, saved through the API

`meta.save_model` calls the writer's `write_model(model, dataset, training)`.
Calling it directly saves a model trained in Python. Here an FM is trained on
two categorical contexts that guide 5 derives from the timestamps: `daypart`
(night, morning, afternoon, evening, in US Central time) and `weekend`.

In [6]:
ratings = pd.read_csv(folder / "ratings.tsv", sep="\t")
local = pd.to_datetime(ratings.timestamp, unit="s", utc=True).dt.tz_convert(
    "America/Chicago"
)
ratings["daypart"] = pd.cut(
    local.dt.hour,
    bins=[0, 6, 12, 18, 24],
    right=False,
    labels=["night", "morning", "afternoon", "evening"],
).astype(str)
ratings["weekend"] = np.where(local.dt.dayofweek >= 5, "weekend", "weekday")
ratings.to_csv(RUNS / "ratings-context.tsv", sep="\t", index=False)
print(Path("configs/context.yml").read_text())

# A context-aware model (FM) over two categorical contexts derived from the
# timestamps, as guide 5 builds them. Trained and saved through the Python API.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../runs/17-serving/ratings-context.tsv
  rating_type: implicit
  labels:
    context_labels: [daypart, weekend]
  dtypes:
    context_types:
      daypart: str
      weekend: str
splitter:
  test_splitting:
    strategy: temporal_leave_k_out
    k: 1
models:
  FM:
    embedding_size: 32
    reg_weight: 0.0001
    batch_size: 2048
    epochs: 3
    learning_rate: 0.001
    neg_samples: 1
    optimization:
      device: cpu
      num_workers: 0
evaluation:
  top_k: [10]
  metrics: [nDCG, HitRate]
  mask_seen: pair
general:
  device: cpu



Training follows guide 5. The test metrics are computed with the `Evaluator`
(guide 12) and passed to `write_model` as `training`, so this card reports them
like the pipeline's. Without `dataset`, `write_model` saves the weights only,
and such a file serves without masking or popularity fallback.

In [7]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.data.writer import LocalWriter
from warprec.evaluation.evaluator import Evaluator
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration
from warprec.utils.registry import model_registry

config = load_design_configuration("configs/context.yml")
context_data, _, _ = initialize_datasets(
    ReaderFactory.get_reader(config), WarpRecCallback(), config
)
fm = model_registry.get(
    "FM",
    params=config.models["FM"],
    info=context_data.info(),
    interactions=context_data.train_set,
    transactions=context_data.train_transactions,
    seed=42,
)
L.Trainer(
    max_epochs=fm.epochs,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
).fit(fm, fm.get_dataloader(context_data.train_set, context_data.train_session))

evaluator = Evaluator(
    list(config.evaluation.metrics),
    list(config.evaluation.top_k),
    train_set=context_data.train_set.get_sparse(),
    mask_seen=config.evaluation.mask_seen,
)
evaluator.evaluate(
    fm, context_data.get_contextual_evaluation_dataloader(), "full", context_data
)
writer = LocalWriter(dataset_name="ml-100k-context", local_path=str(RUNS))
writer.write_model(
    fm,
    context_data,
    training={
        "dataset": "ml-100k with dayparts",
        "evaluation": {"strategy": "full"},
        "metrics": evaluator.compute_results(),
    },
)
fm_payload = torch.load(
    RUNS / f"ml-100k-context/serialized/{fm.name_param}.pth", weights_only=False
)["serving"]
print(fm.name_param, fm_payload["training"]["metrics"])
fm_payload["context_stats"]

FM_BlazingSpiritedBeluga {'nDCG@10': 0.03736498951911926, 'HitRate@10': 0.0733262449502945}


{'daypart': {'type': 'token',
  'counts': {'afternoon': 41222,
   'evening': 28618,
   'morning': 19087,
   'night': 10130}},
 'weekend': {'type': 'token', 'counts': {'weekday': 74526, 'weekend': 24531}}}

`context_stats` counts every context value in training; the server lists the
values most frequent first and refuses any other.

## 2. The serving configuration

The serving configuration is its own file, not a section of the training one.
`server` says where and how to listen; each entry of `endpoints` serves one
checkpoint under a name.

In [8]:
print(Path("configs/serve.yml").read_text())

# Three models on one server. Paths are relative to the directory the server
# is started from (this guide's folder).
server:                   # host: 127.0.0.1 by default; an export listens on 0.0.0.0
  port: 8117
  api_key: guide-17        # every request but /healthz sends X-API-Key: guide-17
  mcp: true                # MCP tools under /mcp (needs warprec[mcp])

endpoints:
  - name: ease
    checkpoint: ../runs/17-serving/ml-100k/serialized/EASE_WiseFoxOfForce.pth
    description: Films from MovieLens-100K, for users seen in training
    item_noun: movie
    unknown_user: popular  # a user not in training gets the most popular films
    item_metadata: &movies
      path: ../data/ml-100k/items.tsv
      sep: "\t"
      header: true
      id_column: item_id
      name_column: title
      columns:
        genres: {column: genres, separator: "|"}
    batching:
      max_batch_size: 32
      batch_wait_timeout_s: 0.005

  - name: sasrec
    checkpoint: ../runs/17-serving/ml-100k/serial

What each endpoint key does:

- `item_metadata` reads `items.tsv`: `title` becomes each item's `name`, and
  `columns` adds `genres` as an attribute, split on `|`. With a header, columns
  are given by name; without one, by position. The `&movies` / `*movies` YAML
  anchor gives all three endpoints the same catalogue.
- `unknown_user: popular` answers a user who is not in the training data with
  the most interacted items, instead of a 404. Only `ease` sets it.
- `description` and `item_noun` are how the endpoint describes itself in its
  card and to MCP clients.
- `batching` groups concurrent requests into one forward pass: at most
  `max_batch_size` requests, waiting at most `batch_wait_timeout_s` for the
  batch to fill.

Two `server` keys need a word:

- `api_key` is required in the `X-API-Key` header of every request but
  `GET /healthz`. The `WARPREC_API_KEY` environment variable overrides it.
- `ray_address` is left unset: the server starts a Ray instance of its own and
  shuts it down when it stops. `auto` (the cluster this machine belongs to) or
  an address joins an existing cluster instead, and on stopping the server
  deletes only its own application.

`load_serving_configuration` is what the command reads the file with. It checks
that every checkpoint and catalogue file exists, so a stale checkpoint name
fails here. Unset keys get their defaults, and `deployment_options()` shows
what each endpoint hands to Ray Serve:

In [9]:
from warprec.utils.config.serving_configuration import (
    ServingConfiguration,
    load_serving_configuration,
)

serving = load_serving_configuration("configs/serve.yml")
pd.DataFrame(
    {
        e.name: {
            "default_k / max_k": f"{e.default_k} / {e.max_k}",
            "mask_seen": e.mask_seen,
            "unknown_user": e.unknown_user,
            "device": e.device,
            "batching": f"{e.batching.max_batch_size}, {e.batching.batch_wait_timeout_s}s",
            "Ray Serve options": e.deployment_options(),
        }
        for e in serving.endpoints
    }
).T

,default_k / max_k,mask_seen,unknown_user,device,batching,Ray Serve options
ease,10 / 100,True,popular,cpu,"32, 0.005s","{'num_replicas': 1, 'max_ongoing_requests': 32}"
sasrec,10 / 100,True,error,cpu,"64, 0.005s","{'num_replicas': 1, 'max_ongoing_requests': 64}"
fm,10 / 100,True,error,cpu,"64, 0.005s","{'num_replicas': 1, 'max_ongoing_requests': 64}"


A replica accepts as many requests at once as a batch holds
(`max_ongoing_requests`): Ray Serve's own default of 5 would cap every batch at
5. Every section rejects keys it does not know, so a typo fails at startup
rather than falling back to a default silently:

In [10]:
from pydantic import ValidationError

raw = yaml.safe_load(Path("configs/serve.yml").read_text())
raw["endpoints"][0]["unknown_users"] = raw["endpoints"][0].pop("unknown_user")
raw["endpoints"][1]["device"] = "cuda:1"
try:
    ServingConfiguration.model_validate(raw)
except ValidationError as error:
    for problem in error.errors():
        print(".".join(map(str, problem["loc"])), "->", problem["msg"][:150])

endpoints.0.unknown_users -> Extra inputs are not permitted
endpoints.1.device -> Value error, device 'cuda:1' cannot be honoured: Ray shows each replica only the GPUs it assigned, as cuda:0. Use device: cuda, and choose GPUs with d


The second error is a rule of serving on Ray: a replica sees only the GPUs Ray
assigned to it, always as `cuda:0`. GPUs are requested with `device: cuda` and
`deployment.ray_actor_options` (`num_gpus`, which can be a fraction).

## 3. Starting the server

`python -m warprec.serve -c configs/serve.yml` (or `warprec.serve -c ...` once
installed) blocks until it is stopped, so the notebook starts it as a
subprocess, with its output in a log file. It answers `/healthz` once its HTTP
gateway is up, and logs `Serving ... at` when every model is ready.

In [11]:
server_log = open(RUNS / "server.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "warprec.serve", "-c", "configs/serve.yml"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
BASE = "http://127.0.0.1:8117"


def log_lines(*patterns):
    """The lines of the server log that contain one of the patterns."""
    text = ANSI.sub("", (RUNS / "server.log").read_text())
    return [
        re.sub(r"^.* - WarpRec - \S+ ", "", line)
        for line in text.splitlines()
        if any(p in line for p in patterns)
    ]


start = time.time()
while not log_lines("Serving ease"):
    if server.poll() is not None:
        raise RuntimeError("\n".join(log_lines("Error", "error")))
    if time.time() - start > 300:
        raise TimeoutError("the server did not start in 5 minutes")
    time.sleep(1)
print(requests.get(f"{BASE}/healthz", timeout=10).json())
print(log_lines("Serving ease")[0])

{'status': 'ok'}
Serving ease, sasrec, fm at http://127.0.0.1:8117/v1/models


## 4. The REST API

Every route lives under `/v1/models`. Two small helpers send a request with the
API key and show the items of an answer as a table.

In [12]:
KEY = {"X-API-Key": "guide-17"}


def call(path, body=None, headers=KEY, **params):
    """GET a route (no body) or POST a JSON body to it; return status and JSON."""
    if body is None:
        r = requests.get(BASE + path, headers=headers, params=params, timeout=60)
    else:
        r = requests.post(BASE + path, headers=headers, json=body, timeout=60)
    return r.status_code, r.json()


def table(entries):
    """Items of an answer as rows: id, name, score or interactions, genres."""
    return pd.DataFrame(
        [
            {
                "item_id": e["item_id"],
                "name": e.get("name"),
                **({"score": round(e["score"], 3)} if "score" in e else {}),
                **({"interactions": e["interactions"]} if "interactions" in e else {}),
                "genres": "|".join(e.get("attributes", {}).get("genres", [])),
            }
            for e in entries
        ]
    )

### What is served

`GET /v1/models` lists every endpoint with its card. `kind` is `general` or
`sequential`; a context-aware model is `general` with the context fields it
needs. `needs_user` is true for sequential models that mix a user embedding
into the sequence (Caser, FOSSIL, STAN), which cannot answer an anonymous
session.

In [13]:
status, models = call("/v1/models")
listing = pd.DataFrame(models)
listing["context"] = [list(c) if c else None for c in listing.context]
listing[["name", "model", "kind", "n_users", "n_items", "needs_user", "context"]]

,name,model,kind,n_users,n_items,needs_user,context
0,ease,EASE,general,943,1678,False,None
1,sasrec,SASRec,sequential,943,1678,False,None
2,fm,FM,general,943,1680,False,"[daypart, weekend]"


MovieLens-100K has 1,682 films. `ease` and `sasrec` know 1,678: four films are
rated only in the held-out validation or test rows, so they never reached
training and cannot be recommended. `fm`'s split holds out only the test row,
and it knows 1,680.

`GET /v1/models/{name}` is one card. It tells a client how to ask, gives a
request that works as is, summarises the catalogue and repeats the training
facts from the checkpoint. SASRec's:

In [14]:
status, card = call("/v1/models/sasrec")
for sentence in card["how_to_ask"]:
    print("-", sentence)
print("example_request:", card["example_request"])
print("catalogue:", json.dumps(card["catalogue"])[:300], "...")
print("training metrics:", card["training"]["metrics"])

- Send user_id: one of the 943 users seen in training.
- An unknown user is refused.
- Or send history: the movies of a session, oldest first, by id or exact name.
- Optional: k, how many movies to return (1 to 100, default 10), and exclude, movies to leave out.
- Optional: filter, to keep only movies with given attributes (genres), such as {"genres": "<value>"}; a list matches any of its values.
- Optional: explain: true adds to each movie the movies of the request that training users most often consumed with it - evidence from the data, not the model's reasoning.
example_request: {'k': 5, 'history': ['Star Wars (1977)', 'Fargo (1996)', 'Contact (1997)']}
catalogue: {"names": true, "n_items": 1678, "attributes": {"genres": {"examples": ["Drama", "Comedy", "Action", "Thriller", "Romance", "Adventure", "Children's", "Crime", "Sci-Fi", "Horror"], "n_values": 19}}} ...
training metrics: {'nDCG@10': 0.07221100479364395, 'HitRate@10': 0.14027629792690277}


### Recommending for a known user

Ids are the dataset's own, as numbers or strings. The full answer for two
items shows the shape: each item has its `name` and `attributes` from the
catalogue, and `fallback` says whether popularity answered instead of the model.

In [15]:
status, answer = call("/v1/models/ease/recommend", {"user_id": 1, "k": 2})
print(json.dumps(answer, indent=1))

{
 "model": "ease",
 "items": [
  {
   "item_id": 423,
   "score": 0.6131627559661865,
   "name": "E.T. the Extra-Terrestrial (1982)",
   "attributes": {
    "genres": [
     "Children's",
     "Drama",
     "Fantasy",
     "Sci-Fi"
    ]
   }
  },
  {
   "item_id": 357,
   "score": 0.5744114518165588,
   "name": "One Flew Over the Cuckoo's Nest (1975)",
   "attributes": {
    "genres": [
     "Drama"
    ]
   }
  }
 ],
 "fallback": false
}


A sequential model answers the same request from the user's stored history
(the `histories` of the checkpoint). Both lists leave out every film user 1
rated in training (`mask_seen`, on by default):

In [16]:
status, by_ease = call("/v1/models/ease/recommend", {"user_id": 1, "k": 5})
status, by_sasrec = call("/v1/models/sasrec/recommend", {"user_id": 1, "k": 5})
pd.concat(
    {"ease": table(by_ease["items"]), "sasrec": table(by_sasrec["items"])}, axis=1
)

ease                                                 \
  item_id                                    name  score   
0     423       E.T. the Extra-Terrestrial (1982)  0.613   
1     357  One Flew Over the Cuckoo's Nest (1975)  0.574   
2     568                            Speed (1994)  0.546   
3     276                Leaving Las Vegas (1995)  0.544   
4     403                           Batman (1989)  0.525   

                                    sasrec                                     \
                            genres item_id                               name   
0  Children's|Drama|Fantasy|Sci-Fi     405         Mission: Impossible (1996)   
1                            Drama     546                Broken Arrow (1996)   
2          Action|Romance|Thriller     423  E.T. the Extra-Terrestrial (1982)   
3                    Drama|Romance     276           Leaving Las Vegas (1995)   
4     Action|Adventure|Crime|Drama     483                  Casablanca (1942)   

                                           
   score                           genres  
0  4.623         Action|Adventure|Mystery  
1  4.257                  Action|Thriller  
2  3.918  Children's|Drama|Fantasy|Sci-Fi  
3  3.603                    Drama|Romance  
4  3.583                Drama|Romance|War

### An anonymous session

Sent a `history` instead of a `user_id`, a sequential model scores that
session, oldest item first. With a catalogue, items can be given by exact name,
matched ignoring case. The session's items are left out of the answer.

In [17]:
session = {
    "history": ["Toy Story (1995)", "aladdin (1992)", "Lion King, The (1994)"],
    "k": 5,
}
status, answer = call("/v1/models/sasrec/recommend", session)
table(answer["items"])

,item_id,name,score,genres
0,174,Raiders of the Lost Ark (1981),6.053,Action|Adventure
1,204,Back to the Future (1985),5.595,Comedy|Sci-Fi
2,56,Pulp Fiction (1994),5.535,Crime|Drama
3,172,"Empire Strikes Back, The (1980)",5.031,Action|Adventure|Drama|Romance|Sci-Fi|War
4,79,"Fugitive, The (1993)",5.012,Action|Thriller


### Scoring your own candidates

`score` returns the model's score of the items sent, in their order, with
nothing masked: it re-ranks candidates another system retrieved. Here three
films for user 1, one of them by name.

In [18]:
status, answer = call(
    "/v1/models/ease/score", {"user_id": 1, "items": [50, 181, "Heat (1995)"]}
)
table(answer["scores"])

,item_id,name,score,genres
0,50,Star Wars (1977),1.142,Action|Adventure|Romance|Sci-Fi|War
1,181,Return of the Jedi (1983),0.986,Action|Adventure|Romance|Sci-Fi|War
2,273,Heat (1995),0.320,Action|Crime|Thriller


### Searching the catalogue

`GET /items?q=` finds the items whose name contains the text, ignoring case,
with the number of training interactions of each. An empty `matches` answers
"does the model know this film?" with no.

In [19]:
for query in ["star trek", "Kung Fu Panda"]:
    status, found = call("/v1/models/ease/items", q=query, limit=4)
    print(f"{query!r}: {len(found['matches'])} matches")
table(call("/v1/models/ease/items", q="star trek", limit=4)[1]["matches"])

'star trek': 4 matches
'Kung Fu Panda': 0 matches


,item_id,name,interactions,genres
0,222,Star Trek: First Contact (1996),363,Action|Adventure|Sci-Fi
1,228,Star Trek: The Wrath of Khan (1982),244,Action|Adventure|Sci-Fi
2,230,Star Trek IV: The Voyage Home (1986),194,Action|Adventure|Sci-Fi
3,229,Star Trek III: The Search for Spock (1984),169,Action|Adventure|Sci-Fi


`POST /items/lookup` takes ids or exact names. What it cannot find comes back
under `unknown` with the closest names: "did you mean".

In [20]:
status, found = call(
    "/v1/models/ease/items/lookup", {"items": ["heat (1995)", 260, "Toy Stroy (1995)"]}
)
print(table(found["items"]).to_string(index=False))
found["unknown"]

 item_id                 name  interactions                         genres
     273          Heat (1995)           217          Action|Crime|Thriller
     260 Event Horizon (1997)           121 Action|Mystery|Sci-Fi|Thriller


[{'query': 'Toy Stroy (1995)',
  'suggestions': ['Toy Story (1995)', 'Tommy Boy (1995)', 'Rob Roy (1995)']}]

### Popular items and filters

`popular` ranks by training interactions, the same for everyone. `filter`
keeps only items with the given attribute values (case-insensitive; a list
matches any of its values; several attributes must all match).

In [21]:
status, answer = call(
    "/v1/models/ease/popular", {"k": 5, "filter": {"genres": "Comedy"}}
)
table(answer["items"])

,item_id,name,score,interactions,genres
0,294,Liar Liar (1997),472.0,472,Comedy
1,1,Toy Story (1995),441.0,441,Animation|Children's|Comedy
2,204,Back to the Future (1985),344.0,344,Comedy|Sci-Fi
3,173,"Princess Bride, The (1987)",323.0,323,Action|Adventure|Comedy|Romance
4,151,Willy Wonka and the Chocolate Factory (1971),323.0,323,Adventure|Children's|Comedy


The same filter on `recommend` keeps the model's ranking and drops what does
not match: user 1's top films among film-noir and westerns.

In [22]:
status, answer = call(
    "/v1/models/ease/recommend",
    {"user_id": 1, "k": 5, "filter": {"genres": ["Film-Noir", "Western"]}},
)
table(answer["items"])

,item_id,name,score,genres
0,435,Butch Cassidy and the Sundance Kid (1969),0.326,Action|Comedy|Western
1,470,Tombstone (1993),0.298,Western
2,510,"Magnificent Seven, The (1954)",0.226,Action|Drama|Western
3,642,"Grifters, The (1990)",0.225,Crime|Drama|Film-Noir
4,302,L.A. Confidential (1997),0.211,Crime|Film-Noir|Mystery|Thriller


`explain: true` adds to each item a `because` list: the request's own items
(the user's training items, or the session) that training users most often
consumed together with it, at most two, with the count. That is co-occurrence
evidence from the training data, not the model's reasoning, which WarpRec
models do not expose: present it as "people who watched X also watched this".

In [23]:
status, answer = call(
    "/v1/models/ease/recommend",
    {
        "user_id": 1,
        "k": 3,
        "filter": {"genres": ["Film-Noir", "Western"]},
        "explain": True,
    },
)
for entry in answer["items"]:
    evidence = [(b["name"], b["co_occurrences"]) for b in entry["because"]]
    print(f"{entry['name']:42}", evidence)

Butch Cassidy and the Sundance Kid (1969)  [('Star Wars (1977)', 182), ('Raiders of the Lost Ark (1981)', 179)]
Tombstone (1993)                           [('Raiders of the Lost Ark (1981)', 97), ('Back to the Future (1985)', 95)]
Magnificent Seven, The (1954)              [('Star Wars (1977)', 102), ('Raiders of the Lost Ark (1981)', 100)]


The counts are the number of training users who have both films, which the
checkpoint's `seen` matrix gives directly:

In [24]:
seen = ease_file["serving"]["seen"].tocsc()
items = ease_file["info"]["item_mapping"]
first = answer["items"][0]
for b in first["because"]:
    both = seen[:, items[first["item_id"]]].multiply(seen[:, items[b["item_id"]]])
    print(f"{first['name']} & {b['name']}: {both.nnz} users")

Butch Cassidy and the Sundance Kid (1969) & Star Wars (1977): 182 users
Butch Cassidy and the Sundance Kid (1969) & Raiders of the Lost Ark (1981): 179 users


A session gets the same evidence from its own items. Here two films sent to
`sasrec`:

In [25]:
status, answer = call(
    "/v1/models/sasrec/recommend",
    {"history": ["Star Wars (1977)", "Fargo (1996)"], "k": 3, "explain": True},
)
for entry in answer["items"]:
    evidence = [(b["name"], b["co_occurrences"]) for b in entry["because"]]
    print(f"{entry['name']:42}", evidence)

Return of the Jedi (1983)                  [('Star Wars (1977)', 463), ('Fargo (1996)', 337)]
Godfather, The (1972)                      [('Star Wars (1977)', 346), ('Fargo (1996)', 287)]
Toy Story (1995)                           [('Star Wars (1977)', 370), ('Fargo (1996)', 318)]


### Unknown users

User 5000 is not in MovieLens-100K. `ease` was configured with
`unknown_user: popular` and answers with the most popular films, marked
`fallback: true`; `sasrec` keeps the default, `error`, and answers 404.

In [26]:
status, answer = call("/v1/models/ease/recommend", {"user_id": 5000, "k": 3})
print(status, "fallback:", answer["fallback"], [e["name"] for e in answer["items"]])
print(*call("/v1/models/sasrec/recommend", {"user_id": 5000, "k": 3}))

200 fallback: True ['Star Wars (1977)', 'Fargo (1996)', 'Contact (1997)']
404 {'detail': "User '5000' is not in the training data."}


### A context-aware model

`GET /context` lists the context fields with their accepted values, most
frequent in training first, their configured descriptions and an example.

In [27]:
status, context = call("/v1/models/fm/context")
print(json.dumps(context, indent=1))

{
 "model": "fm",
 "fields": {
  "daypart": {
   "type": "token",
   "values": [
    "afternoon",
    "evening",
    "morning",
    "night"
   ],
   "counts": {
    "afternoon": 41222,
    "evening": 28618,
    "morning": 19087,
    "night": 10130
   },
   "description": "the part of the day, in US Central time"
  },
  "weekend": {
   "type": "token",
   "values": [
    "weekday",
    "weekend"
   ],
   "counts": {
    "weekday": 74526,
    "weekend": 24531
   },
   "description": "whether it is a weekday or the weekend"
  }
 },
 "example": {
  "daypart": "afternoon",
  "weekend": "weekday"
 }
}


Every request to `fm` carries a `context`, as a dictionary by field or a list in
field order. The same user, at two times:

In [28]:
morning = {"daypart": "morning", "weekend": "weekday"}
evening = ["evening", "weekend"]
answers = {
    str(ctx): table(
        call("/v1/models/fm/recommend", {"user_id": 1, "k": 5, "context": ctx})[1][
            "items"
        ]
    )[["name", "score"]]
    for ctx in (morning, evening)
}
pd.concat(answers, axis=1)

{'daypart': 'morning', 'weekend': 'weekday'}         \
                                          name  score   
0                             Liar Liar (1997)  2.594   
1                  English Patient, The (1996)  2.591   
2                                Scream (1996)  2.470   
3                         Air Force One (1997)  2.398   
4                      Schindler's List (1993)  2.197   

        ['evening', 'weekend']         
                          name  score  
0                Scream (1996)  2.185  
1  English Patient, The (1996)  2.031  
2             Liar Liar (1997)  2.002  
3   Mission: Impossible (1996)  1.978  
4         Air Force One (1997)  1.932

The context moves the scores and the order: Liar Liar leads on a weekday
morning, Scream on a weekend evening. Seen-item masking stays per user, not per
context: a film user 1 rated in any context is left out of both lists.

### Errors that explain themselves

A wrong or missing key is a 401. A request the model cannot answer is a 422 or
a 404 whose `detail` says why and, where it can, what would work:

In [29]:
print(*call("/v1/models", headers={}))
bad_requests = {
    "ease": [
        {"user_id": 1, "filter": {"genres": "Comdy"}},
        {"history": [1, 2]},
        {"user_id": 1, "k": 1000},
    ],
    "sasrec": [{"history": ["Toy Stroy (1995)"]}],
    "fm": [
        {"user_id": 1},
        {"user_id": 1, "context": {"daypart": "brunch", "weekend": "weekend"}},
    ],
}
for name, bodies in bad_requests.items():
    for body in bodies:
        status, answer = call(f"/v1/models/{name}/recommend", body)
        print(f"{status} {name}: {answer['detail']}")

401 {'detail': 'Missing or invalid X-API-Key header.'}
422 ease: No item has genres 'Comdy'. Did you mean 'Comedy'?
422 ease: EASE is not sequential: send a user_id, not a history.
422 ease: k must be between 1 and 100, got 1000.


422 sasrec: Item 'Toy Stroy (1995)' is not in the model's catalogue. Did you mean 'Toy Story (1995)', 'Tommy Boy (1995)', 'Rob Roy (1995)'?


422 fm: FM is context-aware: send a context with the fields ['daypart', 'weekend'].
422 fm: 'brunch' is not a known value of context 'daypart'. Known values: ['afternoon', 'evening', 'morning', 'night'].


A field the request schema does not know is refused before it reaches the
model, with FastAPI's own 422, which names the field:

In [30]:
status, answer = call("/v1/models/ease/recommend", {"user": 1})
print(status, [(e["loc"], e["msg"]) for e in answer["detail"]])

422 [(['body', 'user'], 'Extra inputs are not permitted')]


## 5. MCP

With `server.mcp: true` the same server speaks the Model Context Protocol under
`/mcp/`, behind the same API key. An MCP client gets instructions on connecting
(what the server serves, one line per endpoint from its `description`) and a set
of tools that mirror the REST routes. fastmcp, installed with the `mcp` extra,
provides a client; a notebook can `await` it directly.

In [31]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

transport = StreamableHttpTransport(f"{BASE}/mcp/", headers=KEY)


async def show_tools():
    """Connect, print the instructions' first paragraph and every tool."""
    async with Client(transport) as client:
        print(client.initialize_result.instructions.split("\n\n")[0])
        print()
        for tool in await client.list_tools():
            print(f"{tool.name:17}", tool.description.splitlines()[0])


await show_tools()  # noqa: F704 (a notebook cell can await)

This server recommends items with models trained by WarpRec. Each model is an endpoint you name in every tool call:
- ease: Films from MovieLens-100K, for users seen in training
- sasrec: Films from MovieLens-100K, next film from a sequence of films
- fm: Films from MovieLens-100K, ranked for the time of day and week

list_models       List the models on this server: what each serves and what it needs.
describe_model    Describe one model: how to ask it, with an example request that
describe_context  List the context a context-aware model accepts: each field, its
search_items      Search the items a model knows by part of their name, ignoring case.
get_items         Look items up by id or exact name: their name, attributes and
popular_items     The items with the most training interactions - what is popular,
recommend         Recommend items with one of the models from list_models.
score_items       Score given items for a user or a session, in the order sent - to


An agent calls the tools with JSON arguments, naming the endpoint as `model`.
Here it checks the catalogue for a film, then asks the sequential model for
what follows it:

In [32]:
async def ask_agent_style():
    """Search the catalogue, then recommend from what was found."""
    async with Client(transport) as client:
        found = await client.call_tool(
            "search_items", {"model": "sasrec", "query": "godfather"}
        )
        print([(m["item_id"], m["name"]) for m in found.data["matches"]])
        answer = await client.call_tool(
            "recommend",
            {"model": "sasrec", "history": ["Godfather, The (1972)"], "k": 3},
        )
        print([e["name"] for e in answer.data["items"]])


await ask_agent_style()  # noqa: F704

[(127, 'Godfather, The (1972)'), (187, 'Godfather: Part II, The (1974)')]
['Star Wars (1977)', 'Return of the Jedi (1983)', 'Fargo (1996)']


To give the tools to Claude Code, register the server once (from a terminal,
while it runs):

```bash
claude mcp add --transport http warprec http://127.0.0.1:8117/mcp/ --header "X-API-Key: guide-17"
```

Clients configured by JSON take
`{"mcpServers": {"warprec": {"url": "http://127.0.0.1:8117/mcp/", "headers": {"X-API-Key": "guide-17"}}}}`.
The server also offers the model list and cards as resources
(`warprec://models`, `warprec://models/{name}`) and two prompts,
`recommend_for_me` and `explore_catalogue`.

## 6. Exporting for a cluster

On a Ray cluster or KubeRay, an application is described by a Ray Serve config
file rather than started by a process. `--export` writes one from the same
`serve.yml` and exits without serving:

In [33]:
export = subprocess.run(
    [
        sys.executable,
        "-m",
        "warprec.serve",
        "-c",
        "configs/serve.yml",
        "--export",
        str(RUNS / "serve_app.yaml"),
    ],
    capture_output=True,
    text=True,
)
print(re.sub(r"(?m)^.* - WarpRec - \S+ ", "", ANSI.sub("", export.stdout)).strip())
exported = yaml.safe_load((RUNS / "serve_app.yaml").read_text())
application = exported["applications"][0]
print(
    yaml.safe_dump(
        {
            **{k: v for k, v in exported.items() if k != "applications"},
            "applications": [
                {
                    **{k: v for k, v in application.items() if k != "args"},
                    "args": {
                        "config": {"server": application["args"]["config"]["server"]}
                    },
                }
            ],
        },
        sort_keys=False,
    )
)
print(
    "endpoint 1 checkpoint:",
    application["args"]["config"]["endpoints"][0]["checkpoint"],
)

The API key is not written to the exported file. Set WARPREC_API_KEY in the environment of the cluster that runs it.
Ray Serve config written to ../runs/17-serving/serve_app.yaml
proxy_location: EveryNode
http_options:
  host: 0.0.0.0
  port: 8117
applications:
- name: warprec
  route_prefix: /
  import_path: warprec.serving.app:app_builder
  args:
    config:
      server:
        host: 0.0.0.0
        port: 8117
        route_prefix: /
        api_key: null
        mcp: true
        ray_address: null

endpoint 1 checkpoint: guides/runs/17-serving/ml-100k/serialized/EASE_WiseFoxOfForce.pth


The file runs `warprec.serving.app:app_builder` with the whole serving
configuration inlined under `args`. Paths are made absolute, so the checkpoints
must exist at those paths wherever the replicas run. The proxy listens on
`0.0.0.0`, reachable from outside its machine or pod, because `serve.yml` leaves
`server.host` unset (an explicit `host` is kept). The API key is left out: set `WARPREC_API_KEY` in the cluster's environment.
`serve deploy serve_app.yaml` deploys it on a running cluster, and its
`applications` section is what a KubeRay `RayService` takes as
`serveConfigV2`. [Deploying Served Models](https://warprec.readthedocs.io/en/latest/cloud/deployment/)
covers Docker, Kubernetes, Ray clusters and KubeRay, with the image and
manifests.

## 7. Stopping the server

Ctrl-C at a terminal, or `SIGTERM` from a process manager, stops the server
cleanly: it takes the application down and shuts down the Ray instance it
started. Here the signal goes to the subprocess started in section 3.

In [34]:
server.send_signal(signal.SIGINT)
print("exit code:", server.wait(timeout=120))
server_log.close()
print(*log_lines("Received SIGINT", "Serving stopped"), sep="\n")

exit code: 0
Received SIGINT, stopping.
Serving stopped.


## Summary

| Key | What it does | Docs |
|---|---|---|
| `meta.save_model` | train pipeline: write the best model with its serving payload (seen items, histories, context vocabulary, training facts); `writer.write_model(model, dataset, training)` from Python | [Serving Models](https://warprec.readthedocs.io/en/latest/serving/) |
| `server.host`, `port`, `route_prefix` | where the HTTP proxy listens | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `server.api_key` | required `X-API-Key` on every route but `/healthz`; `WARPREC_API_KEY` overrides it | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `server.mcp` | MCP tools under `/mcp/` (needs `warprec[mcp]`) | [Serving Models](https://warprec.readthedocs.io/en/latest/serving/#mcp) |
| `server.ray_address` | unset: a Ray instance of the server's own, started and stopped with it; `auto` or an address: join a cluster and, on stopping, delete only this application | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].name`, `checkpoint` | the URL name and the `.pth` file it serves | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].item_metadata` | item names (`name_column`) and attributes (`columns`, with `separator`) from a delimited file | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].unknown_user` | `error` (404) or `popular` (`fallback: true`) | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].default_k`, `max_k`, `mask_seen` | list lengths and seen-item masking | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].description`, `item_noun`, `context_descriptions` | how the endpoint describes itself in its card and to MCP clients | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].batching` | `max_batch_size` and `batch_wait_timeout_s` of the batched forward pass | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `endpoints[].device`, `deployment` | where a replica runs; replicas, autoscaling and resources, passed to Ray Serve | [Serving Configuration](https://warprec.readthedocs.io/en/latest/configuration/serving/) |
| `--export PATH` | write a Ray Serve config for `serve deploy` or KubeRay instead of serving | [Deploying Served Models](https://warprec.readthedocs.io/en/latest/cloud/deployment/) |